In [10]:
# Import libraries and set project root path
import pandas as pd
import numpy as np
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Base path:", BASE)

Base path: /Users/alexia/Documents/CASA/Dissertation


In [11]:
# Reload all datasets so this notebook runs independently without 01_data_loading.ipynb

# TS007A
ts007a = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/census2021-ts007a-lsoa.csv"))
ts007a = ts007a.rename(columns={
    "geography code": "lsoa_code", "geography": "lsoa_name", "Age: Total": "age_total",
    "Age: Aged 4 years and under": "age_0_4", "Age: Aged 5 to 9 years": "age_5_9",
    "Age: Aged 10 to 14 years": "age_10_14", "Age: Aged 15 to 19 years": "age_15_19",
    "Age: Aged 20 to 24 years": "age_20_24", "Age: Aged 25 to 29 years": "age_25_29",
    "Age: Aged 30 to 34 years": "age_30_34", "Age: Aged 35 to 39 years": "age_35_39",
    "Age: Aged 40 to 44 years": "age_40_44", "Age: Aged 45 to 49 years": "age_45_49",
    "Age: Aged 50 to 54 years": "age_50_54", "Age: Aged 55 to 59 years": "age_55_59",
    "Age: Aged 60 to 64 years": "age_60_64", "Age: Aged 65 to 69 years": "age_65_69",
    "Age: Aged 70 to 74 years": "age_70_74", "Age: Aged 75 to 79 years": "age_75_79",
    "Age: Aged 80 to 84 years": "age_80_84", "Age: Aged 85 years and over": "age_85plus",
}).drop(columns=["date"])

# IoD2025 File 7
iod_raw = pd.read_csv(os.path.join(BASE, "03_data/demand/IoD2025.csv"))
cols_keep = [
    "LSOA code (2021)", "LSOA name (2021)",
    "Local Authority District code (2024)", "Local Authority District name (2024)",
    "Index of Multiple Deprivation (IMD) Score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
    "Income Score (rate)", "Income Rank (where 1 is most deprived)",
    "Income Decile (where 1 is most deprived 10% of LSOAs)",
]
iod2025 = iod_raw[cols_keep].rename(columns={
    "LSOA code (2021)": "lsoa_code", "LSOA name (2021)": "lsoa_name",
    "Local Authority District code (2024)": "lad_code",
    "Local Authority District name (2024)": "lad_name",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
})

# OpenStreetEV GLA
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))
osev = osev_raw[["country_co","id","external_l","name","address","postal_cod","state",
                  "coordinate","coordina_1","location_c","location_1"]].rename(columns={
    "country_co": "country_code", "id": "location_id", "external_l": "external_uuid",
    "name": "location_name", "postal_cod": "postcode", "state": "borough",
    "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "location_1": "location_type",
})

# Zapmap (join_august2025)
zapmap_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/join_august2025.csv"), low_memory=False)
cols_keep = ["location_id","evse_id","address","city","state","coordinate","coordina_1",
             "location_c","zapmap_device_uid","power_band","device_max_power",
             "uid","charing_start","status","charging_duration","month","year","day"]
zapmap = zapmap_raw[cols_keep].rename(columns={
    "state": "borough", "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "charing_start": "charging_start", "uid": "session_uid",
})

print("All datasets reloaded successfully.")
print(f"ts007a: {ts007a.shape}, iod2025: {iod2025.shape}, osev: {osev.shape}, zapmap: {zapmap.shape}")

All datasets reloaded successfully.
ts007a: (35672, 21), iod2025: (33755, 10), osev: (23045, 11), zapmap: (215037, 18)


In [12]:
# Create output directory for cleaned data if it does not exist
output_dir = os.path.join(BASE, "05_processed")
os.makedirs(output_dir, exist_ok=True)
print("Output directory:", output_dir)

Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed


## TS007A: filter London + calculate Pi

In [13]:
# Define all 33 London boroughs (32 boroughs + City of London)
london_boroughs = [
    "City of London", "Barking and Dagenham", "Barnet", "Bexley", "Brent",
    "Bromley", "Camden", "Croydon", "Ealing", "Enfield", "Greenwich",
    "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow", "Havering",
    "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
    "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
    "Redbridge", "Richmond upon Thames", "Southwark", "Sutton",
    "Tower Hamlets", "Waltham Forest", "Wandsworth", "Westminster"
]

import re

def is_london(name, boroughs):
    if not isinstance(name, str):
        return False
    # Match borough name followed by space + digits (LSOA name format), avoids 'Brent' matching 'Brentwood'
    return any(re.match(rf"^{re.escape(b)}\s+\d", name) for b in boroughs)

ts007a_london = ts007a[ts007a["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)

print("TS007A London rows:", len(ts007a_london))

TS007A London rows: 4994


In [14]:
# Compute Pᵢ: driving-age population (17+)
# Aged 15-19 band split proportionally (3/5) to approximate ages 17-19, assuming uniform
# distribution within the band. Necessary because TS007 single-year age is not released at
# LSOA level (ONS statistical disclosure control) — to be documented as a proposal limitation.
age_20plus_cols = ["age_20_24","age_25_29","age_30_34","age_35_39","age_40_44",
                    "age_45_49","age_50_54","age_55_59","age_60_64","age_65_69",
                    "age_70_74","age_75_79","age_80_84","age_85plus"]

ts007a_london["driving_age_pop"] = (
    ts007a_london["age_15_19"] * (3 / 5) + ts007a_london[age_20plus_cols].sum(axis=1)
).round(0).astype(int)

census_london = ts007a_london[["lsoa_code", "lsoa_name", "age_total", "driving_age_pop"]].copy()

print("=== Census London (TS007A-derived) ===")
print("Shape:", census_london.shape)
print(census_london.head(3).to_string())
print()
print("driving_age_pop range:", census_london["driving_age_pop"].min(), "–", census_london["driving_age_pop"].max())
print("driving_age_pop / age_total ratio (sanity check):",
      (census_london["driving_age_pop"].sum() / census_london["age_total"].sum()).round(3))

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/census_london_clean.csv")
census_london.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Census London (TS007A-derived) ===
Shape: (4994, 4)
   lsoa_code            lsoa_name  age_total  driving_age_pop
0  E01000001  City of London 001A       1473             1346
1  E01000002  City of London 001B       1384             1293
2  E01000003  City of London 001C       1613             1500

driving_age_pop range: 778 – 3293
driving_age_pop / age_total ratio (sanity check): 0.796

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/census_london_clean.csv


## Task 1 — IoD2025 → London filter

### filter + match check

In [18]:
# Filter IoD2025 to Greater London LSOAs using the 2021 codes already confirmed in census_london.
# Both datasets are native LSOA 2021 boundaries, so unlike the old IMD2019 pipeline, no
# 2011→2021 lookup or imputation is needed — expect a clean 1:1 match.
london_codes = set(census_london["lsoa_code"])

iod2025_london = iod2025[iod2025["lsoa_code"].isin(london_codes)].reset_index(drop=True)

matched  = len(iod2025_london)
expected = len(london_codes)
missing  = london_codes - set(iod2025_london["lsoa_code"])

print("=== IoD2025 London ===")
print(f"Matched: {matched} / Expected: {expected}")
print(f"Missing LSOAs: {len(missing)}")
if missing:
    print("Sample missing codes:", list(missing)[:10])
print()
print(iod2025_london.head(3).to_string())

=== IoD2025 London ===
Matched: 4994 / Expected: 4994
Missing LSOAs: 0

   lsoa_code            lsoa_name   lad_code        lad_name  imd_score  imd_rank  imd_decile  income_score  income_rank  income_decile
0  E01000001  City of London 001A  E09000001  City of London      8.742     26525           8         0.013        33730             10
1  E01000002  City of London 001B  E09000001  City of London      4.722     31203          10         0.018        33669             10
2  E01000003  City of London 001C  E09000001  City of London      9.250     25913           8         0.107        25167              8


### validate value ranges

In [19]:
# Validate income_score range against proposal's documented range [0.002–0.998]
print("income_score stats:")
print(iod2025_london["income_score"].describe())
print()
print("income_decile value counts:")
print(iod2025_london["income_decile"].value_counts().sort_index())

income_score stats:
count    4994.000000
mean        0.270942
std         0.147900
min         0.010000
25%         0.152000
50%         0.262000
75%         0.372000
max         0.998000
Name: income_score, dtype: float64

income_decile value counts:
income_decile
1     513
2     837
3     838
4     676
5     502
6     412
7     338
8     290
9     265
10    323
Name: count, dtype: int64


### save imd_london_clean.csv

In [20]:
output_path = os.path.join(BASE, "05_processed/imd_london_clean.csv")
iod2025_london.to_csv(output_path, index=False)
print("Saved to:", output_path)
print("Shape:", iod2025_london.shape)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/imd_london_clean.csv
Shape: (4994, 10)


## Task 2 — Zapmap → datetime parsing + duration cleaning

### parse + filter

In [21]:
# Parse charging_start to datetime
zapmap["charging_start"] = pd.to_datetime(zapmap["charging_start"], errors="coerce")
parse_failures = zapmap["charging_start"].isna().sum()
print(f"charging_start parse failures: {parse_failures}")

# Drop rows with invalid charging_duration: <= 0 or > 10,080 minutes (7-day observation window)
rows_before = len(zapmap)
zapmap_clean = zapmap[
    (zapmap["charging_duration"] > 0) &
    (zapmap["charging_duration"] <= 10080) &
    (zapmap["charging_start"].notna())
].copy()
rows_after = len(zapmap_clean)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Dropped:     {rows_before - rows_after} ({(1 - rows_after/rows_before):.1%})")

charging_start parse failures: 0
Rows before: 215037
Rows after:  215037
Dropped:     0 (0.0%)


### add hour/dayofweek + save

In [22]:
# Add temporal columns for EDA (Block C in 03_EDA.ipynb)
zapmap_clean["hour"] = zapmap_clean["charging_start"].dt.hour
zapmap_clean["dayofweek"] = zapmap_clean["charging_start"].dt.dayofweek  # 0 = Monday

print("=== Zapmap Cleaned ===")
print("Shape:", zapmap_clean.shape)
print(zapmap_clean[["evse_id","charging_start","charging_duration","hour","dayofweek"]].head(5).to_string())
print()
print("charging_duration stats:")
print(zapmap_clean["charging_duration"].describe())
print()
print("Unique evse_id:", zapmap_clean["evse_id"].nunique())
print("Unique location_id:", zapmap_clean["location_id"].nunique())

output_path = os.path.join(BASE, "05_processed/zapmap_clean.csv")
zapmap_clean.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Zapmap Cleaned ===
Shape: (215037, 20)
                            evse_id      charging_start  charging_duration  hour  dayofweek
0  174b834d444d9c65929b489ce7e76fb6 2025-08-06 09:28:13              332.0     9          2
1  174b834d444d9c65929b489ce7e76fb6 2025-08-06 10:32:15               88.0    10          2
2  174b834d444d9c65929b489ce7e76fb6 2025-08-06 10:39:45              952.0    10          2
3  174b834d444d9c65929b489ce7e76fb6 2025-08-06 18:12:21              235.0    18          2
4  174b834d444d9c65929b489ce7e76fb6 2025-08-07 06:16:52              119.0     6          3

charging_duration stats:
count    215037.000000
mean        416.071988
std         752.417130
min           1.000000
25%           7.000000
50%         180.000000
75%         448.000000
max        3600.000000
Name: charging_duration, dtype: float64

Unique evse_id: 10465
Unique location_id: 7582

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/zapmap_clean.csv


## OpenStreetEV — duplicate + bounding box check

### check + clean + save

In [23]:
# Greater London approximate bounding box: lat 51.28–51.70, lon -0.51–0.33
print("=== OpenStreetEV Basic Checks ===")
print(f"Total rows: {len(osev)}")
print(f"Duplicate location_id: {osev.duplicated(subset=['location_id']).sum()}")
print(f"Missing latitude: {osev['latitude'].isna().sum()}")
print(f"Missing longitude: {osev['longitude'].isna().sum()}")

outside_bbox = osev[
    (osev["latitude"]  < 51.28) | (osev["latitude"]  > 51.70) |
    (osev["longitude"] < -0.51) | (osev["longitude"] > 0.33)
]
print(f"Rows outside London bounding box: {len(outside_bbox)}")

osev_clean = osev.drop_duplicates(subset=["location_id"], keep="first")
osev_clean = osev_clean[
    (osev_clean["latitude"]  >= 51.28) & (osev_clean["latitude"]  <= 51.70) &
    (osev_clean["longitude"] >= -0.51) & (osev_clean["longitude"] <= 0.33)
].reset_index(drop=True)

print()
print("=== OpenStreetEV Cleaned ===")
print("Shape:", osev_clean.shape)

output_path = os.path.join(BASE, "05_processed/osev_london_clean.csv")
osev_clean.to_csv(output_path, index=False)
print("Saved to:", output_path)

=== OpenStreetEV Basic Checks ===
Total rows: 23045
Duplicate location_id: 30
Missing latitude: 0
Missing longitude: 0
Rows outside London bounding box: 0

=== OpenStreetEV Cleaned ===
Shape: (23015, 11)
Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/osev_london_clean.csv


## Task 3 — Data Processing Summary Table

### pipeline_summary

In [24]:
# Data Processing Summary Table
# Note: "EVSEs matched to LSOA via spatial join" requires ONS LSOA 2021 boundary geometries
# (not loaded in this notebook — needed for Block B in 03_EDA.ipynb). "Candidate sites" rows
# depend on the Camden toy model (Notebook 4), not yet built. Both marked "Pending" — rerun
# this cell once those steps are complete to fill them in.

summary_data = [
    ("LSOAs loaded (Greater London)",                len(census_london)),
    ("On-street EVSE locations (OpenStreetEV_GLA)",   int((osev_clean["location_category"] == "On-Street").sum())),
    ("Session records (join_august2025, post-clean)", len(zapmap_clean)),
    ("Unique EVSEs with ≥1 session",                  zapmap_clean["evse_id"].nunique()),
    ("EVSEs matched to LSOA via spatial join",        "Pending — needs LSOA boundary file (03_EDA)"),
    ("Candidate sites generated (Camden, pre-buffer)","Pending — Notebook 4 (Camden toy model)"),
    ("Candidate sites after 50m exclusion buffer",    "Pending — Notebook 4 (Camden toy model)"),
]

pipeline_summary = pd.DataFrame(summary_data, columns=["Item", "Count"])
print(pipeline_summary.to_string(index=False))

output_path = os.path.join(BASE, "05_processed/pipeline_summary.csv")
pipeline_summary.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

                                          Item                                       Count
                 LSOAs loaded (Greater London)                                        4994
   On-street EVSE locations (OpenStreetEV_GLA)                                       21366
 Session records (join_august2025, post-clean)                                      215037
                  Unique EVSEs with ≥1 session                                       10465
        EVSEs matched to LSOA via spatial join Pending — needs LSOA boundary file (03_EDA)
Candidate sites generated (Camden, pre-buffer)     Pending — Notebook 4 (Camden toy model)
    Candidate sites after 50m exclusion buffer     Pending — Notebook 4 (Camden toy model)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/pipeline_summary.csv


## Final summary of all cleaned files

In [25]:
processed_dir = os.path.join(BASE, "05_processed")
files = [f for f in os.listdir(processed_dir) if f.endswith(".csv")]

print("=== Cleaned Files in 05_processed/ ===\n")
for f in sorted(files):
    path = os.path.join(processed_dir, f)
    df = pd.read_csv(path)
    print(f"{f}")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Columns: {df.columns.tolist()}\n")

=== Cleaned Files in 05_processed/ ===

census_london_clean.csv
  Shape: 4,994 rows × 4 cols
  Columns: ['lsoa_code', 'lsoa_name', 'age_total', 'driving_age_pop']

imd_london_clean.csv
  Shape: 4,994 rows × 10 cols
  Columns: ['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'imd_score', 'imd_rank', 'imd_decile', 'income_score', 'income_rank', 'income_decile']

osev_london_clean.csv
  Shape: 23,015 rows × 11 cols
  Columns: ['country_code', 'location_id', 'external_uuid', 'location_name', 'address', 'postcode', 'borough', 'latitude', 'longitude', 'location_category', 'location_type']

pipeline_summary.csv
  Shape: 7 rows × 2 cols
  Columns: ['Item', 'Count']

zapmap_clean.csv
  Shape: 215,037 rows × 20 cols
  Columns: ['location_id', 'evse_id', 'address', 'city', 'borough', 'latitude', 'longitude', 'location_category', 'zapmap_device_uid', 'power_band', 'device_max_power', 'session_uid', 'charging_start', 'status', 'charging_duration', 'month', 'year', 'day', 'hour', 'dayofweek']

